# Hierarchical clustering

_Machine Learning_

**Merge the closest groups, over and over, into a tree of clusters.**

Hierarchical clustering builds a tree of nested groups.

     The common bottom-up style is called agglomerative.

---

This notebook builds hierarchical clustering up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We'll cluster three well-separated blobs and check how faithfully different *linkage rules* recover the true groups. We build it in three steps: (1) make the data, (2) run agglomerative clustering under each linkage, (3) score each against the ground truth.

### Step 1 — Make three labelled blobs

`make_blobs` draws 300 points from three Gaussian clouds. Because the clouds are tight (`cluster_std=0.7`) and well separated, a bottom-up merge has a fair shot at recovering them.

We keep `true_labels` only to *score* the clustering afterwards — the algorithm itself never sees them.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs

# Draw 300 points from 3 tight, well-separated Gaussian clouds.
X, true_labels = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.7,
    random_state=0,
)
print("data shape:", X.shape)

### Step 2 — Cluster under three linkage rules

Agglomerative clustering starts with every point as its own cluster and repeatedly merges the two *closest* clusters. "Closest" depends on the **linkage** rule:

- **ward** merges the pair that least increases within-cluster variance.
- **average** uses the mean distance between all cross-cluster point pairs.
- **complete** uses the farthest pair (the maximum distance).

We ask each for exactly 3 clusters so the results are comparable.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

linkages = ["ward", "average", "complete"]
predicted = {}

for link in linkages:
    agg = AgglomerativeClustering(n_clusters=3, linkage=link)
    labels = agg.fit_predict(X)
    predicted[link] = labels

### Step 3 — Score each linkage against the truth

The **adjusted Rand index (ARI)** compares two labellings while ignoring how the cluster numbers are named: 1.0 is a perfect match, 0.0 is chance. We score every linkage's labels against `true_labels` to see which merge rule best recovers the three blobs.

In [ ]:
from sklearn.metrics import adjusted_rand_score

for link in linkages:
    labels = predicted[link]
    ari = adjusted_rand_score(true_labels, labels)
    print("linkage %-9s  ARI vs truth: %.3f" % (link, ari))

## Visualize the data & results

_Merging nearest wines bottom-up — which linkage best recovers the three real cultivars?_

Now we switch to the real **wine** dataset and visualise two things: where the clusters land in 2-D, and how the linkage choice affects quality. We build it in three steps.

### Step 1 — Standardise the wine data and project to 2-D

The 13 wine features live on very different scales, so we **standardise** them (zero mean, unit variance) before clustering — otherwise large-valued features would dominate the distances. We then use **PCA** to compress the 13 dimensions down to 2 so we can draw the points on a flat plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

wine = load_wine()

# Put every feature on the same scale before measuring distances.
X = StandardScaler().fit_transform(wine.data)

# Compress 13 features down to 2 so we can plot them.
P = PCA(n_components=2, random_state=0).fit_transform(X)
print("projected shape:", P.shape)

### Step 2 — Cluster in 2-D and plot the groups

We run ward-linkage agglomerative clustering on the 2-D projection and colour each point by its assigned cluster. This is the left-hand panel: a visual check of whether the three groups the algorithm found look spatially coherent.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Cluster the 2-D points with ward linkage.
labels = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(P)

colors = ["#4ea1ff", "#7ee787", "#c89bff"]
for c in range(3):
    pts = P[labels == c]
    ax1.scatter(pts[:, 0], pts[:, 1], c=colors[c], edgecolor="k")
ax1.set_xlabel("PCA component 1")
ax1.set_ylabel("PCA component 2")
ax1.set_title("Agglomerative clusters (ward)")

### Step 3 — Compare linkage quality against the true cultivars

The wine dataset comes with true cultivar labels, so we can score each linkage with the ARI again — this time on real data. The right-hand bar chart shows which merge rule best recovers the three cultivars, then we draw both panels.

In [ ]:
from sklearn.metrics import adjusted_rand_score

links = ["ward", "average", "complete"]
aris = []
for l in links:
    labels_l = AgglomerativeClustering(n_clusters=3, linkage=l).fit_predict(P)
    ari = adjusted_rand_score(wine.target, labels_l)
    aris.append(ari)

ax2.bar(links, aris, color=["#7ee787", "#ffb454", "#4ea1ff"])
ax2.set_title("Cluster quality by linkage")
plt.show()